## Problem Statement

Conversation AI to request Recipie for a Food Item  and it gives Grocery List to prepare that item

## Agentic Use Case

The model should be able to access Blinkit website and the grocery List item to cart

## Code

In [1]:
import json
import time
import logging

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("grocery_agent")


MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

if "model" not in globals():
    log.info("Loading tokenizer and model (%s)...", MODEL_ID)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=dtype,  
        device_map="auto",
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    log.info("Model loaded on device=%s dtype=%s", device, dtype)
else:
    log.info("Model already loaded, skipping reload.")

/home/sreekar/nemotron/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-22 14:36:04,834 | INFO | Loading tokenizer and model (Qwen/Qwen2.5-3B-Instruct)...
2026-07-22 14:36:06,746 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-22 14:36:06,747 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-22 14:36:06,784 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json "HTTP/1.1 200 OK"
2026-07-22 14:36:07,012 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/token

## Request LLM to generate prompt

In [2]:
def _generate_once(dish_name: str, max_new_tokens: int) -> list[str]:
    prompt = [
        {
            "role": "system",
            "content": (
                "You are a culinary assistant. Given a dish name, output ONLY "
                "a raw JSON array of strings — the raw grocery ingredients "
                "needed to cook that specific dish, nothing else. No "
                "commentary, no markdown fences. The array content must "
                "match the dish given, not any example."
            ),
        },
        {
            "role": "user",
            "content": (
                'List raw ingredients to buy for making: Margherita Pizza\n'
                'Answer: ["Pizza dough", "Mozzarella cheese", "Tomatoes", "Basil", "Olive oil"]'
            ),
        },
        {"role": "assistant", "content": '["Pizza dough", "Mozzarella cheese", "Tomatoes", "Basil", "Olive oil"]'},
        {"role": "user", "content": f"List raw ingredients to buy for making: {dish_name}"},
    ]

    inputs = tokenizer.apply_chat_template(
        prompt, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy — deterministic, avoids syntax-breaking sampling
            repetition_penalty=1.1,   # mild; 1.3 was too aggressive and warped tokens
            pad_token_id=tokenizer.pad_token_id,
            # no_repeat_ngram_size intentionally removed — it was forcing the
            # model off natural words (into things like "_Kofte_Mix_") once
            # ingredient vocabulary repeated, breaking JSON syntax.
        )
    response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    # Robust-ish JSON array extraction — tolerates a truncated/unclosed array
    # by trimming back to the last complete element rather than failing outright.
    start = response.find("[")
    if start == -1:
        log.warning("No '[' found in model output: %r", response)
        return []

    snippet = response[start:]
    end = snippet.rfind("]")
    if end != -1:
        snippet = snippet[: end + 1]
    else:
        last_complete = snippet.rfind('",')
        if last_complete == -1:
            log.warning("Could not recover a complete JSON array from: %r", response)
            return []
        snippet = snippet[: last_complete + 1] + "]"

    try:
        items = json.loads(snippet)
        seen = set()
        deduped = []
        for i in items:
            i = str(i).strip()
            if i and i.lower() not in seen:
                seen.add(i.lower())
                deduped.append(i)
        if not deduped:
            raise ValueError("Empty ingredient list")
        return deduped
    except Exception as e:
        log.warning("Failed to parse model output (%s): %r", e, response)
        return []


In [3]:

def generate_grocery_list(dish_name: str, max_new_tokens: int = 150, retries: int = 2) -> list[str]:
    """Ask the local LLM for a raw-ingredient shopping list for a dish.
    """
    for attempt in range(retries + 1):
        items = _generate_once(dish_name, max_new_tokens)
        if items:
            return items
        if attempt < retries:
            log.warning("Attempt %d failed for '%s', retrying...", attempt + 1, dish_name)
    return []

## Add Items Driver Code

In [4]:
ADD_BUTTON_XPATHS = [
    '//div[text()="ADD" or text()="Add"]',
    '//div[contains(@class, "AddToCart")]',
    '//button[contains(., "ADD") or contains(., "Add")]',
    '//div[contains(text(),"ADD")]/ancestor::div[1]',
]
def get_driver() -> webdriver.Chrome:
    """Launch (or reuse, if already running in this session) a Chrome driver."""
    global _driver
    if "_driver" in globals() and _driver is not None:
        try:
            _ = _driver.title  # cheap liveness check
            return _driver
        except Exception:
            log.info("Existing driver session is dead, starting a new one.")

    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    service = Service(ChromeDriverManager().install())
    _driver = webdriver.Chrome(service=service, options=options)
    return _driver


In [5]:

def add_items_to_blinkit(items: list[str], wait_for_location: bool = True) -> dict:
    """Search + add each item to the Blinkit cart. Returns a {item: status} report."""
    if not items:
        log.warning("No items to add — skipping browser automation.")
        return {}

    driver = get_driver()
    wait = WebDriverWait(driver, 10)
    results: dict[str, str] = {}

    driver.get("https://blinkit.com")

    if wait_for_location:
        # Blocking input() instead of a blind sleep(15): the cell pauses
        # until you've actually set the location and pressed Enter,
        # instead of racing a fixed timer.
        input(
            "\n👉 Set/confirm your delivery location in the Chrome window, "
            "then press Enter here to continue..."
        )

    for item in items:
        log.info("Processing item: %s", item)
        try:
            search_url = f"https://blinkit.com/s/?q={item.replace(' ', '%20')}"
            driver.get(search_url)

            added = False
            for xpath in ADD_BUTTON_XPATHS:
                try:
                    button = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
                    time.sleep(0.5)
                    driver.execute_script("arguments[0].click();", button)
                    added = True
                    break
                except Exception:
                    continue

            if added:
                log.info("✅ Added '%s' to cart", item)
                results[item] = "added"
            else:
                log.warning("⚠️ Could not find/click ADD button for '%s'", item)
                results[item] = "not_found"

            time.sleep(2)

        except Exception as e:
            log.error("❌ Error adding '%s': %s", item, e)
            results[item] = f"error: {e}"

    log.info("Done. Review your cart in the browser window.")
    return results



In [6]:
def close_driver():
    """Call this explicitly when you're done reviewing the cart."""
    global _driver
    if "_driver" in globals() and _driver is not None:
        _driver.quit()
        _driver = None
        log.info("Driver closed.")


## Run Code

In [8]:
dish = "Malai Kofta"

log.info("Generating grocery list for: %s", dish)
grocery_list = generate_grocery_list(dish)

if not grocery_list:
    log.warning("Model failed to produce a list — check the dish name or retry.")
else:
    log.info("Grocery list: %s", grocery_list)
    report = add_items_to_blinkit(grocery_list, wait_for_location=True)
    print(report)


2026-07-22 14:38:12,367 | INFO | Generating grocery list for: Malai Kofta
2026-07-22 14:38:15,451 | INFO | Grocery list: ['Onion', 'Potato', 'Ginger Garlic Paste', 'Flour', 'Malai (Cream)', 'Besan (Gram Flour)', 'Curry Leaves', 'Green Chilly Powder', 'Turmeric Powder', 'Red Chili Powder', 'Cumin Seeds', 'Mustard Seeds', 'Asafoetida', 'Garam Masala', 'Oil']
2026-07-22 14:38:15,453 | INFO | Existing driver session is dead, starting a new one.
2026-07-22 14:38:15,453 | INFO | ====== WebDriver manager ======
2026-07-22 14:38:15,490 | INFO | Driver [/home/sreekar/.wdm/drivers/chromedriver/linux64/150.0.7871.124/chromedriver-linux64/chromedriver] found in cache by browser version



👉 Set/confirm your delivery location in the Chrome window, then press Enter here to continue... 


2026-07-22 14:38:41,602 | INFO | Processing item: Onion
2026-07-22 14:38:43,583 | INFO | ✅ Added 'Onion' to cart
2026-07-22 14:38:45,586 | INFO | Processing item: Potato
2026-07-22 14:38:47,454 | INFO | ✅ Added 'Potato' to cart
2026-07-22 14:38:49,456 | INFO | Processing item: Ginger Garlic Paste
2026-07-22 14:38:51,171 | INFO | ✅ Added 'Ginger Garlic Paste' to cart
2026-07-22 14:38:53,173 | INFO | Processing item: Flour
2026-07-22 14:38:54,918 | INFO | ✅ Added 'Flour' to cart
2026-07-22 14:38:56,920 | INFO | Processing item: Malai (Cream)
2026-07-22 14:38:58,652 | INFO | ✅ Added 'Malai (Cream)' to cart
2026-07-22 14:39:04,055 | INFO | Processing item: Besan (Gram Flour)
2026-07-22 14:39:05,815 | INFO | ✅ Added 'Besan (Gram Flour)' to cart
2026-07-22 14:39:07,817 | INFO | Processing item: Curry Leaves
2026-07-22 14:39:09,626 | INFO | ✅ Added 'Curry Leaves' to cart
2026-07-22 14:39:11,628 | INFO | Processing item: Green Chilly Powder
2026-07-22 14:39:13,318 | INFO | ✅ Added 'Green Chill

{'Onion': 'added', 'Potato': 'added', 'Ginger Garlic Paste': 'added', 'Flour': 'added', 'Malai (Cream)': 'added', 'Besan (Gram Flour)': 'added', 'Curry Leaves': 'added', 'Green Chilly Powder': 'added', 'Turmeric Powder': 'added', 'Red Chili Powder': 'not_found', 'Cumin Seeds': 'added', 'Mustard Seeds': 'added', 'Asafoetida': 'added', 'Garam Masala': 'added', 'Oil': 'added'}
